In [ ]:
import pandas as pd

In [2]:
df = pd.read_parquet("../data/aggregated/feature_data.parquet")
df.head()

,timestamp,coin_id,price,market_cap,volume,return,return_6,return_12,return_24,ma_deviation_6,...,volatility_12,volatility_24,normalized_momentum_6,normalized_momentum_12,normalized_momentum_24,log_volume_change_6,log_volume_change_12,log_volume_change_24,volume_to_mcap,target
0,2025-05-04 08:00:00+00:00,aave,147.163054,2.223734e+09,1.872286e+08,-0.001078,0.010051,0.020906,0.024367,0.004570,...,0.009368,0.008838,1.473298,2.231755,2.756986,0.083303,0.390049,0.233448,0.080838,1
1,2025-05-04 09:00:00+00:00,aave,145.420187,2.199279e+09,1.855157e+08,-0.011843,-0.000798,-0.004441,0.008598,-0.007196,...,0.009363,0.009198,-0.090549,-0.474287,0.934756,0.051550,0.302192,0.199252,0.080983,1
2,2025-05-04 10:00:00+00:00,aave,145.271404,2.197707e+09,1.871798e+08,-0.001023,-0.005064,-0.025816,0.006712,-0.007376,...,0.006566,0.009202,-0.584583,-3.932120,0.729421,0.047891,0.208043,0.202565,0.081737,1
3,2025-05-04 11:00:00+00:00,aave,144.696797,2.187299e+09,1.798898e+08,-0.003955,-0.004795,-0.024827,0.015505,-0.010517,...,0.006533,0.008838,-0.554829,-3.800448,1.754397,0.000548,0.125311,0.154065,0.079036,1
4,2025-05-04 12:00:00+00:00,aave,144.645778,2.183627e+09,1.769548e+08,-0.000353,-0.019583,-0.017605,0.016343,-0.007598,...,0.006297,0.008832,-4.477139,-2.795620,1.850391,-0.038665,0.068770,0.112060,0.077921,1


In [10]:
df.shape

(791060, 23)

In [4]:
df.groupby("coin_id")["timestamp"].agg(["min", "max", "count"])

,min,max,count
coin_id,,,
aave,2025-05-04 08:00:00+00:00,2026-05-13 15:00:00+00:00,8984
algorand,2025-05-04 08:00:00+00:00,2026-05-13 15:00:00+00:00,8984
aptos,2025-05-04 08:00:00+00:00,2026-05-13 15:00:00+00:00,8984
arbitrum,2025-05-04 08:00:00+00:00,2026-05-13 15:00:00+00:00,8984
aster-2,2025-09-20 02:00:00+00:00,2026-05-13 15:00:00+00:00,5654
...,...,...,...
world-liberty-financial,2025-09-02 13:00:00+00:00,2026-05-13 15:00:00+00:00,6075
worldcoin-wld,2025-05-04 08:00:00+00:00,2026-05-13 15:00:00+00:00,8984
xdce-crowd-sale,2025-05-04 08:00:00+00:00,2026-05-13 15:00:00+00:00,8984


In [5]:
df.columns

Index(['timestamp', 'coin_id', 'price', 'market_cap', 'volume', 'return',
       'return_6', 'return_12', 'return_24', 'ma_deviation_6',
       'ma_deviation_12', 'ma_deviation_24', 'volatility_6', 'volatility_12',
       'volatility_24', 'normalized_momentum_6', 'normalized_momentum_12',
       'normalized_momentum_24', 'log_volume_change_6', 'log_volume_change_12',
       'log_volume_change_24', 'volume_to_mcap', 'target'],
      dtype='str')

In [6]:
features = []

columns = df.columns
for column in columns:
    features.append(column)

print(features)

['timestamp', 'coin_id', 'price', 'market_cap', 'volume', 'return', 'return_6', 'return_12', 'return_24', 'ma_deviation_6', 'ma_deviation_12', 'ma_deviation_24', 'volatility_6', 'volatility_12', 'volatility_24', 'normalized_momentum_6', 'normalized_momentum_12', 'normalized_momentum_24', 'log_volume_change_6', 'log_volume_change_12', 'log_volume_change_24', 'volume_to_mcap', 'target']


In [7]:
df[["coin_id", "timestamp", "price", "target"]]

,coin_id,timestamp,price,target
0,aave,2025-05-04 08:00:00+00:00,147.163054,1
1,aave,2025-05-04 09:00:00+00:00,145.420187,1
2,aave,2025-05-04 10:00:00+00:00,145.271404,1
3,aave,2025-05-04 11:00:00+00:00,144.696797,1
4,aave,2025-05-04 12:00:00+00:00,144.645778,1
...,...,...,...,...
791055,zcash,2026-05-13 11:00:00+00:00,433.548314,0
791056,zcash,2026-05-13 12:00:00+00:00,429.189533,0
791057,zcash,2026-05-13 13:00:00+00:00,431.293580,0
791058,zcash,2026-05-13 14:00:00+00:00,419.464093,0


In [9]:
coin_count = df.groupby("coin_id")["timestamp"].count().sort_values(ascending=False)
coin_count

coin_id
binancecoin       9095
bitcoin           9095
ethereum          9095
ripple            9095
tether            9095
                  ... 
midnight-3        3702
united-stables    3471
figure-heloc      3467
ylds              2720
hashnote-usyc     1462
Name: timestamp, Length: 95, dtype: int64

In [11]:
coin_count[coin_count > 8000].count()

np.int64(80)

In [12]:
df.groupby("target")["target"].count()

target
0    405542
1    385518
Name: target, dtype: int64

In [13]:
target_label_distribution = df.groupby("target")["target"].count()

total = len(df)

for idx, label in target_label_distribution.items():
    print(f"target with label {idx} is represented {round((label / total) * 100, 2)}%")

target with label 0 is represented 51.27%
target with label 1 is represented 48.73%


In [14]:
timestamps = df["timestamp"].sort_values().unique()
timestamps

<DatetimeArray>
['2025-04-29 17:00:00+00:00', '2025-04-29 18:00:00+00:00',
 '2025-04-29 19:00:00+00:00', '2025-04-29 20:00:00+00:00',
 '2025-04-29 21:00:00+00:00', '2025-04-29 22:00:00+00:00',
 '2025-04-29 23:00:00+00:00', '2025-04-30 00:00:00+00:00',
 '2025-04-30 01:00:00+00:00', '2025-04-30 02:00:00+00:00',
 ...
 '2026-05-13 06:00:00+00:00', '2026-05-13 07:00:00+00:00',
 '2026-05-13 08:00:00+00:00', '2026-05-13 09:00:00+00:00',
 '2026-05-13 10:00:00+00:00', '2026-05-13 11:00:00+00:00',
 '2026-05-13 12:00:00+00:00', '2026-05-13 13:00:00+00:00',
 '2026-05-13 14:00:00+00:00', '2026-05-13 15:00:00+00:00']
Length: 9095, dtype: datetime64[us, UTC]

In [15]:
dates = {}

for timestamp in timestamps:
    year = timestamp.year
    month = timestamp.month

    if year not in dates:
        dates[year] = [month]
    if year in dates:
        if month not in dates.get(year):
            dates.get(year).append(month)

dates

{2025: [4, 5, 6, 7, 8, 9, 10, 11, 12], 2026: [1, 2, 3, 4, 5]}

In [16]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.training_pipeline.training_data_builder import TrainingDataBuilder

builder = TrainingDataBuilder()

train_val_df, test_df = builder.split_by_time(df)

print("train_val_df:", train_val_df.shape)
print("test_df:", test_df.shape)
print("train range:", train_val_df["timestamp"].min(), train_val_df["timestamp"].max())
print("test range:", test_df["timestamp"].min(), test_df["timestamp"].max())

train_val_df: (663222, 23)
test_df: (127838, 23)
train range: 2025-04-29 17:00:00+00:00 2026-03-17 18:00:00+00:00
test range: 2026-03-17 19:00:00+00:00 2026-05-13 15:00:00+00:00


In [17]:
df.sort_values(["timestamp", "coin_id"]).groupby("coin_id").tail(1).reset_index(
    drop=True
)

,timestamp,coin_id,price,market_cap,volume,return,return_6,return_12,return_24,ma_deviation_6,...,volatility_12,volatility_24,normalized_momentum_6,normalized_momentum_12,normalized_momentum_24,log_volume_change_6,log_volume_change_12,log_volume_change_24,volume_to_mcap,target
0,2026-04-29 16:00:00+00:00,hashnote-usyc,0.886732,2.306471e+09,8.867316e+04,0.000351,-0.000510,0.005103,0.007086,-0.000015,...,0.001247,0.000935,-0.396772,4.093481,7.575809,-0.000510,9.704262,0.007060,0.000038,0
1,2026-05-13 15:00:00+00:00,aave,75.609678,1.147653e+09,2.587319e+08,0.002837,-0.018160,-0.014100,0.012857,-0.009516,...,0.007390,0.006068,-1.911474,-1.908022,2.119002,-0.006845,-0.040074,-0.084770,0.203304,1
2,2026-05-13 15:00:00+00:00,algorand,0.092479,8.238612e+08,2.728546e+07,-0.004606,-0.035955,-0.046263,-0.040614,-0.019347,...,0.006840,0.007597,-4.835436,-6.763623,-5.346147,-0.102720,-0.291210,-0.301107,0.032582,1
3,2026-05-13 15:00:00+00:00,aptos,0.830911,6.805190e+08,5.440246e+07,-0.000287,-0.043931,-0.032189,-0.002840,-0.023081,...,0.009276,0.008128,-4.155425,-3.470254,-0.349369,-0.010101,-0.018086,-0.122914,0.076908,0
4,2026-05-13 15:00:00+00:00,arbitrum,0.104890,6.452424e+08,5.833287e+07,0.002102,-0.044813,-0.036137,-0.008881,-0.025382,...,0.012135,0.009798,-3.015865,-2.977936,-0.906458,0.066285,0.066153,-0.043998,0.086549,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,2026-05-13 15:00:00+00:00,world-liberty-financial,0.052391,1.665252e+09,4.222511e+07,0.008982,-0.022507,-0.006816,0.003574,-0.002187,...,0.008825,0.009497,-3.170076,-0.772339,0.376341,-0.054043,0.046541,-0.099301,0.025040,1
91,2026-05-13 15:00:00+00:00,worldcoin-wld,0.206943,6.983897e+08,7.662454e+07,-0.001493,-0.058453,-0.034678,0.001609,-0.031540,...,0.013981,0.011263,-3.812991,-2.480462,0.142826,0.303341,0.369955,0.184674,0.104104,0
92,2026-05-13 15:00:00+00:00,xdce-crowd-sale,0.024841,4.952511e+08,1.339547e+07,-0.006058,-0.028569,0.011142,-0.009035,-0.013844,...,0.008491,0.006976,-8.295290,1.312243,-1.295229,-0.342013,-0.341630,-0.347031,0.026689,1
93,2026-05-13 15:00:00+00:00,ylds,0.782928,4.149690e+08,4.220942e+06,0.001181,0.000759,0.003471,0.001534,0.001147,...,0.000865,0.000725,0.692671,4.012296,2.117323,0.376640,0.435688,-0.350276,0.010120,0


In [18]:
online_features = pd.read_parquet("../data/online_store/online_features.parquet")

In [19]:
online_features.shape

(95, 22)

In [20]:
online_features.head()

,timestamp,coin_id,price,market_cap,volume,return,return_6,return_12,return_24,ma_deviation_6,...,volatility_6,volatility_12,volatility_24,normalized_momentum_6,normalized_momentum_12,normalized_momentum_24,log_volume_change_6,log_volume_change_12,log_volume_change_24,volume_to_mcap
0,2026-05-14 15:00:00+00:00,aave,77.267482,1.170024e+09,2.039172e+08,0.018929,0.021938,0.022776,0.021926,0.018221,...,0.008978,0.007885,0.007301,2.443510,2.888566,3.003232,-0.135528,-0.131054,-0.238079,0.160659
1,2026-05-14 15:00:00+00:00,algorand,0.093415,8.315116e+08,2.249032e+07,0.012941,0.021913,0.008980,0.010119,0.015479,...,0.006080,0.007032,0.007105,3.603967,1.277030,1.424244,-0.112921,-0.109911,-0.193269,0.026688
2,2026-05-14 15:00:00+00:00,aptos,0.826230,6.766145e+08,4.805679e+07,0.014882,0.024124,0.017707,-0.005634,0.017652,...,0.007599,0.007873,0.007471,3.174412,2.249223,-0.754102,-0.162080,-0.148718,-0.124026,0.068616
3,2026-05-14 15:00:00+00:00,arbitrum,0.104503,6.409972e+08,4.344282e+07,0.023447,0.026593,0.023223,-0.003688,0.023380,...,0.010600,0.008895,0.008372,2.508909,2.610788,-0.440518,-0.287275,-0.287595,-0.294720,0.065576
4,2026-05-14 15:00:00+00:00,aster-2,0.527938,1.361379e+09,8.735434e+07,0.005846,-0.000659,0.009218,0.004962,0.005187,...,0.005698,0.004765,0.004424,-0.115606,1.934300,1.121499,-0.139674,-0.170411,-0.147014,0.062191


In [21]:
test = online_features.sort_values(by="volume", ascending=False)
test

,timestamp,coin_id,price,market_cap,volume,return,return_6,return_12,return_24,ma_deviation_6,...,volatility_6,volatility_12,volatility_24,normalized_momentum_6,normalized_momentum_12,normalized_momentum_24,log_volume_change_6,log_volume_change_12,log_volume_change_24,volume_to_mcap
76,2026-05-14 15:00:00+00:00,tether,0.782698,1.485322e+11,4.799442e+10,0.001263,0.001379,0.001659,0.000547,0.001227,...,0.000558,0.000634,0.000493,2.470468,2.616436,1.110422,-0.043880,-0.159284,-0.095241,0.279996
9,2026-05-14 15:00:00+00:00,bitcoin,63331.620118,1.268026e+12,2.953413e+10,0.014405,0.015013,0.021230,0.015388,0.014637,...,0.006715,0.005240,0.004793,2.235794,4.051153,3.210420,0.014584,-0.178620,-0.089294,0.023024
24,2026-05-14 15:00:00+00:00,ethereum,1789.140570,2.159017e+11,1.092207e+10,0.014432,0.009832,0.016925,0.008753,0.011290,...,0.006817,0.005456,0.005242,1.442128,3.101954,1.669859,-0.041396,0.012576,0.030334,0.049350
82,2026-05-14 15:00:00+00:00,usd-coin,0.782860,6.007098e+10,9.775057e+09,0.001198,0.001438,0.001696,-0.000196,0.001156,...,0.000489,0.000654,0.000615,2.942999,2.595333,-0.318949,-0.049441,-0.025455,-0.287311,0.150766
72,2026-05-14 15:00:00+00:00,solana,72.189817,4.170171e+10,2.514362e+09,0.013571,0.013883,0.019356,0.005138,0.012911,...,0.006896,0.006337,0.005775,2.013281,3.054305,0.889837,-0.191027,-0.194172,-0.050591,0.058546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26,2026-05-14 15:00:00+00:00,falcon-finance,0.781808,1.393773e+09,1.633168e+05,0.002167,0.001708,0.002066,-0.000492,0.001693,...,0.000945,0.000890,0.000810,1.808142,2.322317,-0.607407,-0.533807,-0.405964,-0.666972,0.000117
34,2026-04-30 16:00:00+00:00,hashnote-usyc,0.877695,2.282988e+09,8.776952e+04,-0.000350,-0.007128,-0.012039,-0.009383,-0.002479,...,0.001290,0.001127,0.001197,-5.527520,-10.686410,-7.836952,-0.007153,-0.705209,-0.009428,0.000038
33,2026-05-14 15:00:00+00:00,hash-2,0.008794,4.659391e+08,2.335676e+04,0.070748,0.074299,0.073421,0.068185,0.034479,...,0.036588,0.025714,0.018567,2.030672,2.855312,3.672451,-0.034556,0.067691,0.523214,0.000050
87,2026-05-14 15:00:00+00:00,usual-usd,0.780683,4.353030e+08,1.469250e+03,0.000224,0.000318,0.000433,-0.000505,0.000269,...,0.000250,0.000508,0.000406,1.270368,0.851848,-1.244587,-3.309137,-3.313281,-3.747538,0.000003


In [23]:
coins = list(online_features["coin_id"].unique())
coins

['aave',
 'algorand',
 'aptos',
 'arbitrum',
 'aster-2',
 'avalanche-2',
 'beldex',
 'bfusd',
 'binancecoin',
 'bitcoin',
 'bitcoin-cash',
 'bitget-token',
 'bittensor',
 'bonk',
 'canton-network',
 'cardano',
 'chainlink',
 'cosmos',
 'crypto-com-chain',
 'dai',
 'dexe',
 'dogecoin',
 'ethena',
 'ethena-usde',
 'ethereum',
 'ethereum-classic',
 'falcon-finance',
 'figure-heloc',
 'filecoin',
 'flare-networks',
 'gatechain-token',
 'gho',
 'global-dollar',
 'hash-2',
 'hashnote-usyc',
 'hedera-hashgraph',
 'htx-dao',
 'hyperliquid',
 'internet-computer',
 'jupiter-exchange-solana',
 'just',
 'kaspa',
 'kucoin-shares',
 'leo-token',
 'litecoin',
 'mantle',
 'memecore',
 'midnight-3',
 'monero',
 'morpho',
 'near',
 'nexo',
 'official-trump',
 'okb',
 'ondo-finance',
 'ondo-us-dollar-yield',
 'ousg',
 'pax-gold',
 'paypal-usd',
 'pepe',
 'pi-network',
 'polkadot',
 'polygon-ecosystem-token',
 'pudgy-penguins',
 'pump-fun',
 'quant-network',
 'rain',
 'render-token',
 'ripple',
 'ripple-u

In [27]:
df.dtypes

timestamp                 datetime64[us, UTC]
coin_id                                   str
price                                 float64
market_cap                            float64
volume                                float64
return                                float64
return_6                              float64
return_12                             float64
return_24                             float64
ma_deviation_6                        float64
ma_deviation_12                       float64
ma_deviation_24                       float64
volatility_6                          float64
volatility_12                         float64
volatility_24                         float64
normalized_momentum_6                 float64
normalized_momentum_12                float64
normalized_momentum_24                float64
log_volume_change_6                   float64
log_volume_change_12                  float64
log_volume_change_24                  float64
volume_to_mcap                    